# è fortemente consigliato runnare il notebook su colab

# Preliminari

In [1]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

In [2]:
import torch
import json
import copy

from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from unsloth.chat_templates import train_on_responses_only
from pathlib import Path
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
from transformers import TextStreamer
from huggingface_hub import login
from google.colab import userdata

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [4]:
# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

# Parametri

In [5]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
model_name = "unsloth/Llama-3.2-3B-Instruct"
TRAIN_FILE_NAME = "data_luca_train-v1.jsonl"
TEST_FILE_NAME = "data_luca_test-v1.jsonl"
VAL_FILE_NAME = "data_luca_val-v1.jsonl"
FINETUNED_MODEL_NAME = "llama-3.2-3B-FT-v1"
RANDOM_STATE = 2025

# Load model and tokenizer

In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name, # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2025.10.1: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.10.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


# Data preparation

In [8]:
# Carichiamo il nostro file JSON
if "COLAB_" not in "".join(os.environ.keys()):
    train_path = Path('../data/ft-dataset/' + TRAIN_FILE_NAME)
    test_path = Path('../data/ft-dataset/' + TEST_FILE_NAME)
    if VAL_FILE_NAME not in ["", None]:
        val_path = Path('../data/ft-dataset/' + VAL_FILE_NAME)
else:
    train_path = TRAIN_FILE_NAME
    test_path = TEST_FILE_NAME
    if VAL_FILE_NAME not in ["", None]:
        val_path = VAL_FILE_NAME

with open(train_path, "r", encoding="utf-8") as f:
    raw_train = [json.loads(line) for line in f]

with open(test_path, "r", encoding="utf-8") as f:
    raw_test = [json.loads(line) for line in f]

if VAL_FILE_NAME not in ["", None]:
    with open(val_path, "r", encoding="utf-8") as f:
        raw_val = [json.loads(line) for line in f]

I dati in `raw_train` sono salvati con la seguente struttura:
```
[
    {
        "messages": [
            {"role": "system", "content": <contenuto di sistema>},
            {"role": "user", "content": <contenuto di user>},
            {"role": "assistant", "content": <contenuto di assistant>}
        ]
    },
    ...
]
```

## Creazione dataset corretto

Trasformiamo `<contenuto di assistant>` in una stringa.

In [9]:
training_data = copy.deepcopy(raw_train)
test_data = copy.deepcopy(raw_test)
val_data = copy.deepcopy(raw_val)

for split in [training_data, test_data, val_data]:
    if split is None:
        continue
    for conversation in split:
        for message in conversation['messages']:
            if message['role'] == 'assistant':
                message['content'] = json.dumps(message['content'])

Ora possiamo applicare il chat template, dato che il contenuto è una stringa

In [10]:
print(type(raw_train[0]['messages'][2]['content']), raw_train[0]['messages'][2]['content'])
print(type(training_data[0]['messages'][2]['content']), training_data[0]['messages'][2]['content'])

<class 'dict'> {'morfologia': 'solido_anulare', 'spessore_parietale': None, 'estensione_cranio_caudale': None, 'distanza_oai': None, 'posizione': 'giunzione', 'carcinosi_peritoneale': None, 'lesioni_ossee': None, 'riflessione_peritoneale_anteriore': 'cavallo', 'infiltrazione_tessuto_adiposo': 'si_5mm_plus', 'infiltrazione_sfinteri': 'no', 'infiltrazione_organi_extra': 'no', 'coinvolgimento_riflessione_peritoneale': 'no', 'coinvolgimento_fascia_mesorettale': 'no', 'linfonodi_sospetti': 10.0, 'linfonodi_mesorettali': True, 'linfonodi_rettali_superiori': True, 'linfonodi_mesenterici_inferiori': False, 'linfonodi_iliaci_interni': True, 'linfonodi_otturatori': True, 'linfonodi_sacrali': False, 'linfonodi_inguinali_sotto_dentata': False, 'linfonodi_inguinali': False, 'linfonodi_iliaci_esterni': True, 'linfonodi_iliaci_comuni': True, 'linfonodi_paraortici': False, 'linfonodi_altri': False, 'depositi_tumorali': 'no', 'numero_depositi': 0.0, 'emvi_esteso': 'no'}
<class 'str'> {"morfologia": "so

Ora possiamo applicare il chat template, dato che il contenuto è una stringa

In [11]:
def formatting_messages(messages: list[dict]) -> str:
    return tokenizer.apply_chat_template(messages,
                                         tokenize=False,
                                         add_generation_prompt=False)

In [12]:
def create_hugging_face_dataset(data: list[dict]) -> Dataset:
    system_content = []
    user_content = []
    assistant_content = []
    formatted_text = []
    for conversation in data:
        formatted_text.append(formatting_messages(conversation['messages']))
        for message in conversation['messages']:
            if message['role'] == 'system':
                system_content.append(message['content'])
            elif message['role'] == 'user':
                user_content.append(message['content'])
            elif message['role'] == 'assistant':
                assistant_content.append(message['content'])
    return Dataset.from_dict(
            {
                'system_content': system_content,
                'user_content': user_content,
                'assistant_content': assistant_content,
                'text': formatted_text
            })

In [13]:
dataset_train = create_hugging_face_dataset(training_data)
dataset_test = create_hugging_face_dataset(test_data)
if VAL_FILE_NAME not in ["", None]:
    dataset_val = create_hugging_face_dataset(val_data)

In [14]:
for split in [dataset_train, dataset_test, dataset_val]:
    if split is None:
        continue
    display(split)

Dataset({
    features: ['system_content', 'user_content', 'assistant_content', 'text'],
    num_rows: 120
})

Dataset({
    features: ['system_content', 'user_content', 'assistant_content', 'text'],
    num_rows: 35
})

Dataset({
    features: ['system_content', 'user_content', 'assistant_content', 'text'],
    num_rows: 18
})

Vediamo come il chat template ha trasformato le conversazioni.

In [15]:
print(dataset_train[0]['text'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 09 Oct 2025

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite Risonanza Magnetica (RM).

Il tuo compito è analizzare attentamente il referto di Risonanza Magnetica fornito e mapparlo nel seguente schema JSON.

Regole di output rigorose:
1. Ogni campo deve contenere ESCLUSIVAMENTE uno dei valori previsti dalle liste/tipologie indicate.
2. Se un dato non è riportato o è ambiguo, inserisci la parola chiave "null" (minuscola).
3. Per i linfonodi, restituisci chiavi booleane ("true"/"false") per ogni sede possibile.
4. NON aggiungere testo, spiegazioni, preamboli o commenti.
5. Utilizza tutte le chiavi fornite nello schema JSON.
6. Restituisci ESCLUSIVAMENTE il JSON finale, assicurandoti che il formato sia VALIDO.

Schema JSON per l'output:

{
  "morfologia": "solido_polipoide | solido_anulare | mucinoso",
  "posizione": "basso | medio | 

<a name="Train"></a>
# Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [16]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_train,
    dataset_text_field = "text",
    eval_dataset=dataset_val,
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3, # Set this for 1 full training run.
        #max_steps = 30,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = RANDOM_STATE,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc,
        eval_strategy='steps',
        push_to_hub=True,
        hub_model_id=FINETUNED_MODEL_NAME
    )
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/120 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/18 [00:00<?, ? examples/s]

### extra

In [17]:
display(trainer.eval_dataset)
display(trainer.train_dataset)

Dataset({
    features: ['system_content', 'user_content', 'assistant_content', 'text', 'input_ids', 'attention_mask'],
    num_rows: 18
})

Dataset({
    features: ['system_content', 'user_content', 'assistant_content', 'text', 'input_ids', 'attention_mask'],
    num_rows: 120
})

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [18]:
print(len(trainer.train_dataset[0]['input_ids']), len(trainer.train_dataset[2]['input_ids']))

2021 1440


In [19]:
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=12):   0%|          | 0/120 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/18 [00:00<?, ? examples/s]

We verify masking is actually done:

In [20]:
print('*--- inizio')
for x, y in zip(trainer.train_dataset[5]['input_ids'][:5], trainer.train_dataset[5]['labels'][:5]):
    print(f"id: {x:<10}label: {y}")
print('\n*--- fine')
for x, y in zip(trainer.train_dataset[5]['input_ids'][-5:], trainer.train_dataset[5]['labels'][-5:]):
    print(f"id: {x:<10}label: {y}")

*--- inizio
id: 128000    label: -100
id: 128000    label: -100
id: 128006    label: -100
id: 9125      label: -100
id: 128007    label: -100

*--- fine
id: 794       label: 794
id: 330       label: 330
id: 6455      label: 6455
id: 9388      label: 9388
id: 128009    label: 128009


In [21]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]  # id di sapce
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

We can see the System and Instruction prompts are successfully masked!

# Training

In [22]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.557 GB.
3.07 GB of memory reserved.


In [23]:
# @title Training
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 120 | Num Epochs = 3 | Total steps = 45
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
1,0.462500,0.459320
2,0.463100,0.452648
3,0.465300,0.423626
4,0.431200,0.370895
5,0.373800,0.288332
6,0.292900,0.206768
7,0.216300,0.171034
8,0.184800,0.151863
9,0.158600,0.130301
10,0.138200,0.105802


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [24]:
# @title Push model to the HUB
trainer.push_to_hub("End of training")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...outputs/training_args.bin: 100%|##########| 6.16kB / 6.16kB            

  ...adapter_model.safetensors:  43%|####3     | 41.9MB / 97.3MB            

  ...nt/outputs/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

CommitInfo(commit_url='https://huggingface.co/iltramont/llama-3.2-3B-FT-v1/commit/ab19889a4f9bb21d4911099fee47c6241587fe94', commit_message='End of training', commit_description='', oid='ab19889a4f9bb21d4911099fee47c6241587fe94', pr_url=None, repo_url=RepoUrl('https://huggingface.co/iltramont/llama-3.2-3B-FT-v1', endpoint='https://huggingface.co', repo_type='model', repo_id='iltramont/llama-3.2-3B-FT-v1'), pr_revision=None, pr_num=None)

In [25]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

259.1367 seconds used for training.
4.32 minutes used for training.
Peak reserved memory = 6.881 GB.
Peak reserved memory for training = 3.811 GB.
Peak reserved memory % of max memory = 17.395 %.
Peak reserved memory for training % of max memory = 9.634 %.


<a name="Inference"></a>
# Inference
Let's run the model! You can change the instruction and input - leave the output blank!



We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [ ]:
test_messages = [
    {'role': 'system', 'content': dataset_test[0]['system_content']},
    {'role': 'user', 'content': dataset_test[0]['user_content']}
]

In [ ]:
tokenized_chat_template = tokenizer.apply_chat_template(
                        test_messages,
                        tokenize = True,
                        add_generation_prompt = True, # Must add for generation
                        return_tensors = "pt"
                        ).to("cuda")

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [ ]:
outputs = model.generate(input_ids=tokenized_chat_template,
                         use_cache=True,
                         temperature=1.5,
                         min_p=0.1)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [ ]:
generated_tokens = outputs[0][len(tokenized_chat_template[0]):-1]
print(tokenizer.decode(generated_tokens))

{"morfologia": "solido_anulare", "spessore_parietale": null, "estensione_cranio_caudale": 500.0, "distanza_oai": null, "posizione": "medio", "spessore_peritoneale": null, "carcinosi_peritoneale": null, "lesioni_ossee": null, "riflessione_peritoneale_anteriore": "cavallo", "infiltrazione_tessuto_adiposo": "no", "infiltrazione_sfinteri": "no", "infiltrazione_organi_extra": null, "coinvolgimento_riflessione_peritoneale": "no", "coinvolgimento_fascia_mesorettale": "no", "linfonodi_sospetti": null.0, "sedi_locoregionali": {"mesorettali": false.0, "rettali_superiori": true.0, "mesenterici_inferiori": false.0, "iliaci_interni": false.0, "otturatori": true.0, "sacrali": false.0, "inguinali_sotto_dentata": false.0.0, "inguinali": 0.0.0, "iliaci_esterni": 0.0.0, "iliaci_comuni": 0.0.0, "paraortici": 0.0.0, "altri": 0.0.0.0}


In [ ]:
prediction = json.loads(tokenizer.decode(generated_tokens))
groundtruth = groundtruth = json.loads(dataset_test[0]['assistant_content'])

print(f"{'LABELS':.^40} {'TRUTH':.^18} {'PREDICTION':.^18}")
for key in groundtruth.keys():
    if key not in ('sedi_locoregionali', 'sedi_non_locoregionali'):
        print(f"{key:.<40} {str(groundtruth[key]):.^18} {str(prediction[key]):.^18}")
    else:
        print(f"{key}")
        for subkey in groundtruth[key].keys():
            print(5*' ' + f"{subkey:.<35} {str(groundtruth[key][subkey]):.^18} {str(prediction[key][subkey]):.^18}")

#display(prediction)
#display(groundtruth)

JSONDecodeError: Expecting ',' delimiter: line 1 column 498 (char 497)

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids=tokenized_chat_template,
                   streamer=text_streamer,
                   use_cache=True,
                   temperature=1.5,
                   min_p=0.1)

{"morfologia": "solido_polipoide", "spessore_parietale": null, "estensione_cranio_caudale": 10.0, "distanza_oai": null, "posizione": "medio", "carcinosi_peritoneale": null, "lesioni_ossee": null, "riflessione_peritoneale_anteriore": null, "infiltrazione_tessuto_adiposo": "si_5mm_plus", "infiltrazione_sfinteri": "no", "infiltrazione_organi_extra": "si", "coinvolgimento_riflessione_peritoneale": "no", "coinvolgimento_fascia_mesorettale": "si", "linfonodi_sospetti": 0.0, "sedi_locoregionali": {"mesorettali": false, "rettali_superiori": true, "mesenterici_inferiori": false, "iliaci_interni": false, "otturatori": true, "sacrali": false, "inguinali_sotto_dentata": false}, "sedi_non_locoregionali": {"inguinali": false, "iliaci_esterni": false, "iliaci_comuni": false, "paraortici": false, "altri": false}, "depositi_tumorali": "no", "numero_depositi": 0.0, "emvi_esteso": "no"}<|eot_id|>


In [ ]:
print(trainer_stats)

TrainOutput(global_step=60, training_loss=0.09714645386363069, metrics={'train_runtime': 954.794, 'train_samples_per_second': 0.503, 'train_steps_per_second': 0.063, 'total_flos': 1.2617869730463744e+16, 'train_loss': 0.09714645386363069, 'epoch': 3.3478260869565215})


<a name="Save"></a>
# Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained(FINETUNED_MODEL_NAME)  # Local saving
tokenizer.save_pretrained(FINETUNED_MODEL_NAME)
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('llama-3.2-3B-FT-v1/tokenizer_config.json',
 'llama-3.2-3B-FT-v1/special_tokens_map.json',
 'llama-3.2-3B-FT-v1/chat_template.jinja',
 'llama-3.2-3B-FT-v1/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if True:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

==((====))==  Unsloth 2025.9.7: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
{"morfologia": "mucinoso", "spessore_parietale": null, "estensione_cranio_caudale": 9.5, "distanza_oai": 6.67, "posizione": "medio", "carcinosi_peritoneale": null, "lesioni_ossee": null, "riflessione_peritoneale_anteriore": "sotto", "infiltrazione_tessuto_adiposo": "si", "infiltrazione_sfinteri": "si", "infiltrazione_organi_extra": "no", "infiltrazione_organi_dettagli": null, "coinvolgimento_riflessione_peritoneale": "si", "coinvolgimento_fascia_mesorettale": "si", "linfonodi_sospetti": 5.0, "sedi_locoregionali": ["mesorettali", "otturato

You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

# Alternative di salvataggio e caricamento

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "", # Get a token at https://huggingface.co/settings/tokens
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
